In [10]:
import numpy as np
import pandas as pd
from pulp import *
import copy

In [48]:
data = pd.read_csv('dataset.csv')
data = data.sort_values(by = 'Unnamed: 0')
ids = data[['Unnamed: 0']]
data = data.drop('Unnamed: 0', axis = 'columns')
data = data.reset_index(drop = True)
data.index = 'a'+ data.index.astype('str')
data

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
a0,3,840000.0,13.8,3.0,2.0,2.0,534.0,136.00,1965.0
a1,4,1100000.0,5.9,4.0,2.0,5.0,559.0,195.00,1920.0
a2,2,495000.0,13.9,2.0,1.0,1.0,76.0,79.00,1980.0
a3,3,1120000.0,7.8,3.0,2.0,1.0,293.0,180.00,2006.0
a4,4,2325000.0,9.2,4.0,3.0,2.0,638.0,314.00,1930.0
a5,3,822000.0,13.0,3.0,1.0,4.0,700.0,105.00,1950.0
a6,3,1560000.0,4.6,3.0,2.0,0.0,198.0,148.00,1910.0
a7,2,650000.0,10.5,2.0,1.0,3.0,620.0,85.00,1950.0
a8,3,1223500.0,7.9,3.0,2.0,2.0,721.0,136.00,1980.0
a9,2,790000.0,11.2,2.0,1.0,2.0,196.0,109.00,1970.0


In [41]:
ids

,Unnamed: 0
0,9084
1,3833
2,8817
3,9047
4,8257
5,6594
6,9838
7,6124
8,11048
9,7588


In [3]:
data.describe()

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
count,50.000000,5.000000e+01,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000
mean,2.940000,1.010540e+06,10.760000,2.920000,1.580000,1.72000,400.440000,146.037400,1966.520000
std,0.977502,4.836915e+05,7.117727,0.965528,0.641745,1.03095,305.460586,67.062725,33.749522
min,1.000000,3.610000e+05,2.100000,1.000000,1.000000,0.00000,0.000000,46.000000,1890.000000
25%,2.000000,6.507500e+05,7.325000,2.000000,1.000000,1.00000,170.000000,105.250000,1942.500000
50%,3.000000,8.785000e+05,9.350000,3.000000,1.500000,2.00000,318.500000,132.000000,1970.000000
75%,3.000000,1.255000e+06,13.975000,3.000000,2.000000,2.00000,599.250000,183.750000,1992.750000
max,6.000000,2.440000e+06,45.900000,6.000000,3.000000,5.00000,1452.000000,314.000000,2013.000000


## **1. UTA algorithm - finding minimal subset of inconsistent constraints** ##

In [4]:
# preference_list - list of pairs (better, worse)
preference_list = [("a4", "a9"), ("a9", "a13"), ("a13", "a4"), ("a4", "a1"), ("a32", "a45"), 
("a27", "a18"), ("a25", "a28"), ("a45", "a50"), ("a43", "a37"), ("a39", "a38"), 
("a27", "a18"), ("a31", "a33"), ("a30", "a42"), ("a40", "a35"), ("a27", "a26"),
("a3", "a11"), ("a6", "a7")]

In [6]:
# criteria_types - list of pairs in form (gain or cost type, is discrete?)
criteria_types = [('gain', True), ('cost', False), 
('cost', False), ('gain', True), ('gain', True), 
('gain', True), ('gain', False), ('gain', False), 
('gain', False)]

In [ ]:
# def distribute_breakpoints(min, max, number_of_breakpoints, type_of_cr):
#     cost_or_gain, is_discrete = type_of_cr


In [ ]:
# def interpolate(u_vals, score, bp)

In [ ]:
def build_uta_model(preference_list, df, criteria_types):
    # 0. Define Linear Problem - we want to minimize number of const that are excluded
    model = LpProblem('uta_model', LpMinimize)
    eps = 0.001

    pairwise_comparisons = copy.deepcopy(preference_list)
    binary_variables = LpVariable.dicts('bin_var', pairwise_comparisons, cat='Binary')
    model += lpSum(list(binary_variables.values()))

    # 1. Add weights const
    weights = []
    for crit in df.columns:
        weights.append(LpVariable(f'w_{crit}', lowBound = 1/(5*len(df.columns)), upBound = 0.5, cat = 'Continuous'))

    # 2. Add norm const
    model += lpSum(weights) == 1

    # 3. Utility constraints
    utility_vars = {}
    for alt in df.index:
        for criterion in df.columns:
            curr_util_name = 'u_'+alt+'_'+criterion
            utility_vars[curr_util_name] = LpVariable(curr_util_name, lowBound=0, cat = 'Continuous')
    
    for i in range(len(criteria_types)):
        # Find best and worst
        if criteria_types[i][0] == 'cost':
            best_idx = np.argmin(df[df.columns[i]])
            worst_idx = np.argmax(df[df.columns[i]])
        else:
            worst_idx = np.argmin(df[df.columns[i]])
            best_idx = np.argmax(df[df.columns[i]])

        model += utility_vars[f'u_{df.index[best_idx]}_{df.columns[i]}'] == 1
        model += utility_vars[f'u_{df.index[worst_idx]}_{df.columns[i]}'] == 0

        # Monocity constraint
        if criteria_types[i][0] == 'cost':
            sorted_alts = np.argsort(df[df.columns[i]])
        else:
            sorted_alts = np.argsort(df[df.columns[i]])[::-1]
        for j in range(len(sorted_alts)-1):
                model += utility_vars[f'u_{df.index[sorted_alts[j]]}_{df.columns[i]}'] >= utility_vars[f'u_{df.index[sorted_alts[j+1]]}_{df.columns[i]}']
    print(model)

    # 4. Add utility functions
    for better, worse in pairwise_comparisons:
        # if criterion can't be met, then 1 is substracted on the right side so that it holds
        model += lpSum([...]) > lpSum(...) - binary_vars[(alt1, alt2)] + eps

    return model

In [52]:
build_uta_model(preference_list, data, criteria_types)

uta_model:
MINIMIZE
1*bin_var_('a13',_'a4') + 1*bin_var_('a25',_'a28') + 1*bin_var_('a27',_'a18') + 1*bin_var_('a27',_'a26') + 1*bin_var_('a3',_'a11') + 1*bin_var_('a30',_'a42') + 1*bin_var_('a31',_'a33') + 1*bin_var_('a32',_'a45') + 1*bin_var_('a39',_'a38') + 1*bin_var_('a4',_'a1') + 1*bin_var_('a4',_'a9') + 1*bin_var_('a40',_'a35') + 1*bin_var_('a43',_'a37') + 1*bin_var_('a45',_'a50') + 1*bin_var_('a6',_'a7') + 1*bin_var_('a9',_'a13') + 0
SUBJECT TO
_C1: w_Bathroom + w_Bedroom2 + w_BuildingArea + w_Car + w_Distance
 + w_Landsize + w_Price + w_Rooms + w_YearBuilt = 1

VARIABLES
0 <= bin_var_('a13',_'a4') <= 1 Integer
0 <= bin_var_('a25',_'a28') <= 1 Integer
0 <= bin_var_('a27',_'a18') <= 1 Integer
0 <= bin_var_('a27',_'a26') <= 1 Integer
0 <= bin_var_('a3',_'a11') <= 1 Integer
0 <= bin_var_('a30',_'a42') <= 1 Integer
0 <= bin_var_('a31',_'a33') <= 1 Integer
0 <= bin_var_('a32',_'a45') <= 1 Integer
0 <= bin_var_('a39',_'a38') <= 1 Integer
0 <= bin_var_('a4',_'a1') <= 1 Integer
0 <= bin

/var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/ipykernel_13305/1086142619.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model += utility_vars[f'u_{df.index[sorted_alts[j]]}_{df.columns[i]}'] >= utility_vars[f'u_{df.index[sorted_alts[j+1]]}_{df.columns[i]}']
/var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/ipykernel_13305/1086142619.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model += utility_vars[f'u_{df.index[sorted_alts[j]]}_{df.columns[i]}'] >= utility_vars[f'u_{df.index[sorted_alts[j+1]]}_{df.columns[i]}']
/var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/ipykernel_13305/1086142619.py:44: FutureW

TypeError: must be real number, not ellipsis